<a href="https://colab.research.google.com/github/AriesConsulting/808s-Auto-Generations-/blob/main/Spatial_Quantum_Engine_2_0_%E2%80%94_3D_Stems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import librosa
import tkinter as tk
from tkinter import filedialog
import math

class SpatialQuantumEngine:
    def __init__(self):
        self.ghost_layer = None
        self.velocity_field = None
        self.decay_rate = 0.90  # How long the gravitational ripples linger
        self.color_phase = 0

    def apply_quantum_physics(self, img, sun_x, sun_y, bass_force, mid_force, chromatic_strength=0.25):
        h, w = img.shape[:2]

        # Initialize velocity field for temporal feedback (The Ripple Memory)
        if self.velocity_field is None:
            self.velocity_field = np.zeros((h, w, 2), dtype=np.float32)

        # 1. Create coordinate grid
        x, y = np.meshgrid(np.arange(w), np.arange(h))

        # 2. Gravitational Well Logic (Driven by Bass)
        dx = x - sun_x
        dy = y - sun_y
        dist = np.sqrt(dx**2 + dy**2) + 1e-6

        # Bass dictates the 'mass' of the sun, pulling pixels inward
        pull_mag = (bass_force * 110) / (dist / 40 + 1)
        current_force_x = (dx / dist) * pull_mag
        current_force_y = (dy / dist) * pull_mag

        # 3. Temporal Feedback Integration
        # The velocity field remembers past movement, creating fluid ripples
        self.velocity_field[:, :, 0] = (self.velocity_field[:, :, 0] + current_force_x * 0.3) * self.decay_rate
        self.velocity_field[:, :, 1] = (self.velocity_field[:, :, 1] + current_force_y * 0.3) * self.decay_rate

        # Apply displacement to coordinates
        flex_x = x - self.velocity_field[:, :, 0]
        flex_y = y - self.velocity_field[:, :, 1]

        # 4. Multi-Channel Chromatic Lens (Driven by Mid-Range)
        # Normalize coordinates for radial distortion
        x_n = (2.0 * flex_x - w) / w
        y_n = (2.0 * flex_y - h) / h
        r_radial = np.sqrt(x_n**2 + y_n**2)

        # Distort R, G, and B at different rates
        output_channels = []
        shifts = [1.12, 1.0, 0.88] # RGB scaling factors
        for i, s_mult in enumerate(shifts):
            # Lens strength increases with mid-range activity
            s = chromatic_strength * s_mult * (1 + mid_force)
            dist_factor = 1.0 + s * (r_radial**2)

            mx = (((x_n * dist_factor) + 1.0) * w / 2.0).astype(np.float32)
            my = (((y_n * dist_factor) + 1.0) * h / 2.0).astype(np.float32)

            # Use BORDER_REFLECT to prevent black edges when gravity pulls hard
            ch_remap = cv2.remap(img[:, :, i], mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            output_channels.append(ch_remap)

        return cv2.merge(output_channels)

    def process_frame(self, frame, audio_data, pan_x, f_idx):
        h, w, _ = frame.shape
        if self.ghost_layer is None:
            self.ghost_layer = np.zeros_like(frame)

        # Extract Frequency Bands
        bass = audio_data['bass']
        mids = audio_data['mids']
        highs = audio_data['highs']

        # 1. Solar Positioning (Stereo Pan)
        target_x = int((w/2) + (pan_x * w/2.5))
        # Subtle vertical float based on mids
        target_y = int((h/2) + (math.sin(f_idx * 0.1) * h/10) * mids)

        # 2. Render the Quantum Sun (Luminosity based on Mids)
        sun_radius = int(20 + (bass * 60) + (mids * 40))
        sun_color = (
            int(127 + 127 * math.sin(f_idx * 0.05)),
            int(127 + 127 * math.cos(f_idx * 0.03)),
            255
        )
        sun_overlay = np.zeros_like(frame)
        cv2.circle(sun_overlay, (target_x, target_y), sun_radius, sun_color, -1)
        blur_amt = int(41 + (30 * mids))
        if blur_amt % 2 == 0: blur_amt += 1
        sun_overlay = cv2.GaussianBlur(sun_overlay, (blur_amt, blur_amt), 0)

        # 3. Motion Ghosting (Temporal persistence of light)
        self.ghost_layer = cv2.addWeighted(self.ghost_layer, 0.85, sun_overlay, 0.3, 0)

        # 4. Apply Physics Warp (Bass Pull + Mid Lens + Velocity Field)
        warped_base = self.apply_quantum_physics(frame, target_x, target_y, bass, mids)

        # Combine Warp with Ghosting
        combined_visual = cv2.add(warped_base, self.ghost_layer)

        # 5. Quantum Dot Grid (High-Frequency Flickering)
        dot_canvas = np.zeros_like(frame)
        res = 20
        for y_step in range(0, h, res):
            for x_step in range(0, w, res):
                d = np.sqrt((x_step-target_x)**2 + (y_step-target_y)**2)
                # Dots pulse and jitter based on Highs
                jitter = int((np.random.rand() - 0.5) * 10 * highs)
                dot_r = max(1, int(10 * (1 - d/w) * highs + (2 * bass)))
                cv2.circle(dot_canvas, (x_step + jitter, y_step + jitter), dot_r, sun_color, -1)

        return np.hstack((combined_visual, dot_canvas))

# --- Audio Analysis & Execution ---
def analyze_audio(path):
    y, sr = librosa.load(path, mono=False)
    # Handle Stereo
    if y.ndim > 1:
        left, right = y[0], y[1]
        y_mono = librosa.to_mono(y)
        rms_l = librosa.feature.rms(y=left)[0]
        rms_r = librosa.feature.rms(y=right)[0]
        pan = (rms_r - rms_l) / (rms_l + rms_r + 1e-6)
    else:
        y_mono = y
        pan = np.zeros(100)

    # Short-Time Fourier Transform for frequency bands
    stft = np.abs(librosa.stft(y_mono))
    freqs = librosa.fft_frequencies(sr=sr)

    # Define Bands
    bass_band = stft[(freqs >= 20) & (freqs <= 150)].mean(axis=0)
    mid_band = stft[(freqs > 150) & (freqs <= 2500)].mean(axis=0)
    high_band = stft[(freqs > 2500)].mean(axis=0)

    # Normalize helpers
    def norm(arr): return (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)

    return {
        'bass': norm(bass_band),
        'mids': norm(mid_band),
        'highs': norm(high_band),
        'pan': pan,
        'time_stamps': len(bass_band)
    }

if __name__ == "__main__":
    root = tk.Tk(); root.withdraw()
    file_path = filedialog.askopenfilename(title="Select Media File")
    if not file_path: exit()

    print("Synthesizing Frequency Landscape...")
    audio_features = analyze_audio(file_path)

    cap = cv2.VideoCapture(file_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter("quantum_fft_output.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (w*2, h))
    engine = SpatialQuantumEngine()

    f_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # Sync index
        a_idx = int((f_idx / total_f) * audio_features['time_stamps'])
        a_idx = min(a_idx, audio_features['time_stamps'] - 1)

        current_audio = {
            'bass': audio_features['bass'][a_idx],
            'mids': audio_features['mids'][a_idx],
            'highs': audio_features['highs'][a_idx]
        }
        cur_pan = audio_features['pan'][a_idx] if isinstance(audio_features['pan'], np.ndarray) else 0

        # Process
        output_frame = engine.process_frame(frame, current_audio, cur_pan, f_idx)

        out.write(output_frame)
        cv2.imshow('Quantum FFT Engine', cv2.resize(output_frame, (1200, 500)))

        f_idx += 1
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Success. Rendered {f_idx} frames.")